In [14]:
import os
import httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

BASE_URL = os.getenv("OPENAI_BASE_URL")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL")

transport = httpx.HTTPTransport(verify=False)
http_client = httpx.Client(transport=transport)

llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    http_client=http_client,
    streaming=False
)

In [15]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

In [16]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=3)
tools = [tool]
llm_with_tools = llm.bind_tools(tools)

In [17]:
from langgraph.prebuilt import ToolNode, tools_condition

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)
graph_builder.add_edge("tools", "chatbot")
graph_builder.set_entry_point("chatbot")

In [18]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

In [19]:
graph = graph_builder.compile(checkpointer=memory)

In [25]:
config = {"configurable": {"thread_id": "1"}}

In [26]:
user_input = "Hi there! My name is Will."

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Hi there! My name is Will.
================================== Ai Message ==================================

Hello again, Will! 🌟 I'm glad to see you! How can I assist you today? 😊


In [27]:
user_input = "Remember my name?"

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Remember my name?
================================== Ai Message ==================================

Yes, Will! I remember your name. How can I assist you today? 😊


In [24]:
user_input = "Remember my name?"
config2 = {"configurable": {"thread_id": "2"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config2,
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Remember my name?
================================== Ai Message ==================================

I don't have the capability to remember names or past interactions. However, I can help you by asking for your name each time we chat! Would you like me to proceed with that?


In [28]:
snapshot = graph.get_state(config)
snapshot

StateSnapshot(values={'messages': [HumanMessage(content='Hi there! My name is Will.', additional_kwargs={}, response_metadata={}, id='c2e1c855-57b3-4777-8528-9b2a3192d7bd'), AIMessage(content='Hello, Will! Nice to meet you. How can I assist you today? 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 1825, 'total_tokens': 1930, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 86, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-8B', 'system_fingerprint': '', 'id': '019975c30b93980489d7b4bd96ca8500', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--f1d9796a-f4f2-44db-851e-3acd7ebd86e4-0', usage_metadata={'input_tokens': 1825, 'output_tokens': 105, 'total_tokens': 1930, 'input_token_details': {}, 'output_token_details': {'reasoning': 86}}), HumanMessage(content='Remember my name?', additiona